# Phase 4A — Milestone 1: Regime-Conditional Evaluation Harness

This notebook walks through and validates **Milestone 1** of Phase 4A: the rolling-window + regime-conditional evaluation harness shipped in this branch.

**Reference documents**
- PRD: [`phase-4a-feature-and-label-redesign.prd.md`](../.claude/prds/phase-4a-feature-and-label-redesign.prd.md)
- Plan: [`phase-4a-milestone-1-regime-harness.plan.md`](../.claude/plans/phase-4a-milestone-1-regime-harness.plan.md)
- Concept doc: [`regime-evaluation.md`](../docs/concepts/regime-evaluation.md)

**Why this notebook exists.** The Phase 3 GBM does not beat the ARIMA baseline OOS over the 23-year panel, so the Phase 4 entry gate is not met and Track A (transformers) is deferred. Phase 4A is the diagnostic subproject that earns the right to Phase 4. Milestone 1 adds a *regime axis* to evaluation so subsequent milestones (label redesign, regime-aware features, feature catalog) can attribute results to specific macro periods rather than a single aggregate Sharpe.

**Sections**
1. Setup and imports
2. Synthetic data substrate
3. `DateRangeDetector` walk-through (macro-era axis)
4. `VIXThresholdDetector` walk-through (volatility axis)
5. Point-in-time invariant demo
6. End-to-end with real data + ARIMA baseline
7. Per-regime metrics demo
8. DM test + Phase 4A gate report
9. Cross-axis demo — era × volatility
10. Mini-Milestone-6 preview — quick GBM real-data slice
11. Harness self-tests visualized
12. What's next (Milestones 2–6)


---
## 1 — Setup and imports


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')


In [ ]:
# Milestone 1 surface area — everything we will exercise in this notebook.
from quant.backtest.regimes import (
    DateRangeDetector,
    RegimeDetector,
    VIXThresholdDetector,
    tag_regimes,
)
from quant.backtest.regime_metrics import (
    compute_regime_metrics,
    phase4a_gate_report,
    regime_dm_test,
)
from quant.backtest.report import (
    format_regime_report,
    print_regime_report,
    regime_summary_table,
)
from quant.backtest.harness import (
    BacktestResult,
    run_portfolio_backtest,
)
from quant.backtest.statistics import diebold_mariano

# Existing models for the real-data demo
from quant.models.arima_baseline import ARIMABaseline
from quant.models.buyandhold_baseline import BuyAndHoldBaseline

print('Milestone 1 surface area imported.')


---
## 2 — Synthetic data substrate

A small reproducible panel for the detector walk-throughs in §3–§5. Real data
arrives in §6.


In [ ]:
rng = np.random.default_rng(20260611)

# Synthetic VIX series — three structurally distinct regimes (low / mid / high).
# Used to exercise VIXThresholdDetector without pulling from the lake yet.
n = 1500
dates = pd.bdate_range('2018-01-02', periods=n)

vix_levels = np.concatenate([
    rng.normal(12.0, 1.5, n // 3),   # low-vol regime
    rng.normal(20.0, 2.0, n // 3),   # mid-vol regime
    rng.normal(30.0, 4.0, n - 2 * (n // 3)),  # high-vol regime
]).clip(8.0, 80.0)

synthetic_vix = pd.Series(vix_levels, index=dates, name='VIXCLS')

print(f'Synthetic VIX series:  {len(synthetic_vix)} bars')
print(f'Range:                 {synthetic_vix.min():.2f} → {synthetic_vix.max():.2f}')
print(f'Date span:             {synthetic_vix.index[0].date()} → {synthetic_vix.index[-1].date()}')


---
## 3 — `DateRangeDetector` walk-through

The era axis — used by the Phase 4A success gate. Default ranges:

| Regime      | Span                       |
|-------------|----------------------------|
| `pre_qe`    | (anything before 2010)     |
| `qe_bull`   | 2010-01-01 → 2019-12-31    |
| `covid`     | 2020-01-01 → 2021-12-31    |
| `rate_cycle`| 2022-01-01 → present       |

The detector is independent of market data — it only consults a fixed date table,
so the point-in-time invariant is trivially satisfied.


In [ ]:
era = DateRangeDetector()

# Assert boundary correctness — same expectations as the test suite.
boundary_cases = [
    ('2005-06-15', 'pre_qe'),
    ('2009-12-31', 'pre_qe'),
    ('2010-01-01', 'qe_bull'),
    ('2019-12-31', 'qe_bull'),
    ('2020-01-01', 'covid'),
    ('2021-12-31', 'covid'),
    ('2022-01-01', 'rate_cycle'),
    ('2024-06-15', 'rate_cycle'),
]
for date_str, expected in boundary_cases:
    actual = era.label(pd.to_datetime([date_str])).iloc[0]
    mark = '✓' if actual == expected else '✗'
    print(f'  {mark}  {date_str} -> {actual:<10} (expected {expected})')
    assert actual == expected, f'Boundary failure at {date_str}'
print('\nAll DateRangeDetector boundary cases pass.')


In [ ]:
# Visualise the era partition along a date axis covering all four regimes.
demo_idx = pd.bdate_range('2005-01-03', '2026-04-21')
demo_labels = era.label(demo_idx)

palette = {
    'pre_qe':     '#A0A0A0',
    'qe_bull':    '#7BB7E0',
    'covid':      '#F2A65A',
    'rate_cycle': '#9F7AEA',
}

fig, ax = plt.subplots(figsize=(11, 1.5))
for regime, color in palette.items():
    mask = (demo_labels == regime).to_numpy()
    if mask.any():
        ax.fill_between(demo_idx[mask], 0, 1, color=color, alpha=0.85, label=regime, linewidth=0)
ax.set_yticks([])
ax.set_ylim(0, 1)
ax.set_xlabel('date')
ax.set_title('DateRangeDetector — era partition (default ranges)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.4), ncol=4, frameon=False)
plt.tight_layout()
plt.show()


---
## 4 — `VIXThresholdDetector` walk-through

The volatility axis — complementary to the era axis. Defaults:

| Regime     | Condition         |
|------------|-------------------|
| `low_vol`  | VIX ≤ 15          |
| `mid_vol`  | 15 < VIX < 25     |
| `high_vol` | VIX ≥ 25          |

Thresholds 15 / 25 approximate the long-run VIX 25th / 75th percentiles since 1990
(CBOE). Anchored before any Phase 4A model run — do not retune to make a model pass.


In [ ]:
vol = VIXThresholdDetector(synthetic_vix)
vol_labels = tag_regimes(synthetic_vix.index, vol)

# Sanity check: every bar gets exactly one of {low_vol, mid_vol, high_vol}.
assert set(vol_labels.unique()) <= {'low_vol', 'mid_vol', 'high_vol'}
print(vol_labels.value_counts().rename('bars').to_frame())


In [ ]:
# Visualise VIX overlaid with regime bands.
fig, ax = plt.subplots(figsize=(11, 3.2))
band_colors = {'low_vol': '#7BB7E0', 'mid_vol': '#FFD580', 'high_vol': '#E07B7B'}

for regime, color in band_colors.items():
    mask = (vol_labels == regime).to_numpy()
    if mask.any():
        ax.fill_between(
            synthetic_vix.index, 0, 90,
            where=mask, color=color, alpha=0.18, linewidth=0, label=f'{regime} band',
        )

ax.plot(synthetic_vix.index, synthetic_vix.values, color='black', linewidth=0.8, label='synthetic VIX')
ax.axhline(15, color='#1F77B4', linestyle='--', linewidth=0.8, label='low threshold (15)')
ax.axhline(25, color='#D62728', linestyle='--', linewidth=0.8, label='high threshold (25)')
ax.set_ylabel('VIX')
ax.set_title('VIXThresholdDetector — synthetic VIX overlaid with regime bands')
ax.legend(loc='upper left', frameon=False, ncol=2)
plt.tight_layout()
plt.show()


### Now load **real** FRED VIX from the lake

Same detector, real data. If the lake has not been ingested locally this cell
falls through with a warning — the rest of §4 still works with the synthetic
series above.


In [ ]:
import quant.storage.catalog as catalog

REAL_VIX_LOADED = False
real_vix = None
try:
    df = catalog.query(f"""
        SELECT timestamp, value
        FROM {catalog.table('macro_fred')}
        WHERE series_id = 'VIXCLS'
        ORDER BY timestamp
    """)
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
    real_vix = (
        df.drop_duplicates('timestamp')
          .set_index('timestamp')['value']
          .astype(float)
          .rename('VIXCLS')
    )
    REAL_VIX_LOADED = True
    print(f'Real VIX loaded:   {len(real_vix):,} bars')
    print(f'Span:              {real_vix.index.min().date()} -> {real_vix.index.max().date()}')
    print(f'Mean / std:        {real_vix.mean():.2f} / {real_vix.std():.2f}')
except Exception as exc:
    print(f'Lake unavailable ({type(exc).__name__}): {exc}')
    print('Falling back to synthetic VIX for sec 6 onwards.')


In [ ]:
# Quick visual check on real VIX (only if loaded).
if REAL_VIX_LOADED:
    real_labels = tag_regimes(real_vix.index, VIXThresholdDetector(real_vix))
    print('\nReal-VIX regime counts:')
    print(real_labels.value_counts().rename('bars').to_frame())

    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(real_vix.index, real_vix.values, color='black', linewidth=0.6, label='VIXCLS (FRED)')
    ax.axhline(15, color='#1F77B4', linestyle='--', linewidth=0.8)
    ax.axhline(25, color='#D62728', linestyle='--', linewidth=0.8)
    ax.set_ylabel('VIX')
    ax.set_title('Real VIXCLS overlaid with detector thresholds')
    plt.tight_layout()
    plt.show()
else:
    print('Skipped (no real VIX loaded).')


---
## 5 — Point-in-time invariant demo

> A regime label assigned to date *D* must use only information available at or
> before *D*. Otherwise the model's measured per-regime Sharpe silently inflates.

Two checks:

1. Mutating *future* VIX values does not change the label assigned to a *past* date.
2. Asking the detector to label a date with no corresponding VIX value raises —
   no silent forward-fill, no NaN propagation.


In [ ]:
# Build a VIX series that increases monotonically - future values strictly higher.
vix_series = pd.Series(
    [10.0, 20.0, 30.0],
    index=pd.bdate_range('2020-01-02', periods=3),
    name='VIXCLS',
)
det = VIXThresholdDetector(vix_series)
day0 = vix_series.index[0]

label_before = det.label(pd.DatetimeIndex([day0])).iloc[0]
print(f'Day-0 label with original VIX series: {label_before!r}')
assert label_before == 'low_vol', 'Day 0 should be low_vol because vix[day0]=10 <= 15'

# If the detector were peeking at future VIX, mutating future values would shift
# the day-0 label. With the point-in-time invariant, mutation has no effect.
peek_attempt = VIXThresholdDetector(
    vix_series.copy().replace(20.0, 100.0).replace(30.0, 100.0)
)
label_after = peek_attempt.label(pd.DatetimeIndex([day0])).iloc[0]
print(f'Day-0 label after mutating future bars to 100: {label_after!r}')
assert label_after == label_before, 'Detector peeked into the future!'
print('\n✓ Point-in-time invariant holds for VIXThresholdDetector.')


In [ ]:
# Missing-VIX-date must raise - no silent fill.
missing_dates = pd.bdate_range('2099-01-01', periods=2)
try:
    det.label(missing_dates)
    raise AssertionError('Expected ValueError for missing VIX dates')
except ValueError as exc:
    print(f'✓ Missing VIX correctly raised ValueError:\n  {exc}')


---
## 6 — End-to-end with real data + ARIMA baseline

A small real-data slice — 5 symbols × ~8 years — sized for fast iteration
(~30 seconds), not for the final Phase 4A gate evaluation. The point is to
confirm `BacktestResult.oos_returns` and `oos_forecast_errors` are populated
and aligned, and that `compute_regime_metrics` slices them correctly.

The actual Phase 4A gate evaluation runs on the full 33-symbol × 23-year panel
in Milestone 6.


In [ ]:
DEMO_SYMBOLS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'SPY']
DEMO_START = '2018-01-02'
DEMO_END = '2026-04-21'

REAL_OHLCV_LOADED = False
prices_by_sym = {}
try:
    syms_sql = ', '.join(f"'{s}'" for s in DEMO_SYMBOLS)
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, adjClose, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
          AND timestamp BETWEEN '{DEMO_START}' AND '{DEMO_END}'
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp']).dt.tz_localize(None)
    eq = eq.set_index('timestamp')
    for s in DEMO_SYMBOLS:
        sdf = eq[eq['symbol'] == s][['open', 'high', 'low', 'close', 'volume']].copy()
        sdf = sdf[~sdf.index.duplicated(keep='last')].sort_index()
        if not sdf.empty:
            prices_by_sym[s] = sdf
    REAL_OHLCV_LOADED = bool(prices_by_sym)
    print(f'Loaded OHLCV for {len(prices_by_sym)} symbols')
    for s, df in prices_by_sym.items():
        print(f'  {s}: {len(df):,} bars  {df.index.min().date()} -> {df.index.max().date()}')
except Exception as exc:
    print(f'Lake unavailable ({type(exc).__name__}): {exc}')


In [ ]:
# Minimal features (1 column suffices for ARIMA - it ignores X) and labels.
# For Milestone 1 the model isn't the point; the harness plumbing is.
HORIZON = 1
labels_by_sym, features_by_sym = {}, {}

if REAL_OHLCV_LOADED:
    for s, p in prices_by_sym.items():
        fwd = p['close'].shift(-HORIZON) / p['close'] - 1.0
        fwd = fwd.dropna()
        features_by_sym[s] = pd.DataFrame({'placeholder': 0.0}, index=fwd.index)
        labels_by_sym[s] = fwd
        prices_by_sym[s] = p.loc[fwd.index]
    n_bars = sum(len(v) for v in features_by_sym.values())
    print(f'Built features + labels: {n_bars:,} (sym, bar) pairs.')
else:
    print('Skipping sec 6-10 - no real data loaded.')


In [ ]:
arima_result = None
if REAL_OHLCV_LOADED:
    arima_result = run_portfolio_backtest(
        model=ARIMABaseline(),
        features_by_symbol=features_by_sym,
        labels_by_symbol=labels_by_sym,
        prices_by_symbol=prices_by_sym,
        train_window=504,
        test_window=63,
        step=63,
        label_horizon=HORIZON,
        embargo=3,
    )
    print('ARIMA portfolio backtest complete.')
    print(f'  OOS bars:                {len(arima_result.oos_returns):,}')
    print(f'  OOS forecast-error bars: {len(arima_result.oos_forecast_errors):,}')
    print(f'  Aggregate OOS Sharpe:    {arima_result.oos_metrics["sharpe"]:+.3f}')
    print(f'  Aggregate Max DD:        {arima_result.oos_metrics["max_drawdown"]:+.2%}')

    # Verify the Milestone 1 fields are populated AND aligned.
    assert isinstance(arima_result.oos_returns, pd.Series) and len(arima_result.oos_returns) > 0
    assert arima_result.oos_returns.index.equals(arima_result.oos_forecast_errors.index)
    print('\n✓ BacktestResult.oos_returns and oos_forecast_errors populated and aligned.')


---
## 7 — Per-regime metrics demo

The OOS series from §6 is now tagged with the era detector. `compute_regime_metrics`
groups by regime; `format_regime_report` formats per-regime Sharpe, Sortino, drawdown,
and bar count using the same formatting conventions as the existing `format_report`.


In [ ]:
if arima_result is not None:
    era_labels = tag_regimes(arima_result.oos_returns.index, DateRangeDetector())
    print('Regime composition of the ARIMA OOS series:')
    print(era_labels.value_counts().rename('bars').to_frame())

    per_regime = compute_regime_metrics(arima_result.oos_returns, era_labels)
    print('\nPer-regime Sharpe:')
    for regime, m in per_regime.items():
        print(f'  {regime:<12} Sharpe={m["sharpe"]:+.3f}  '
              f'MaxDD={m["max_drawdown"]:+.2%}  '
              f'TotRet={m["total_return"]:+.2%}')


In [ ]:
if arima_result is not None:
    print(format_regime_report(arima_result, era_labels))
    display(regime_summary_table(arima_result, era_labels))


---
## 8 — DM test + Phase 4A gate report

Compare ARIMA vs `BuyAndHoldBaseline` per regime. Note: `BuyAndHoldBaseline.predict()`
returns a constant `+1`, so the *signal* is always long, and the *raw forecast*
(`+1` constant) is what's used for the forecast error. The DM test on these two
error series is therefore well-defined but trivially degenerate — in regimes where
variance is zero the DM call returns `None` (logged via `warnings.warn`).

For Milestone 1 the goal is to show the pipeline end-to-end; the Phase 4A gate is
calibrated against ARIMA, not against buy-and-hold, so we don't expect a meaningful
"pass" here — the demo is about plumbing, not the verdict.


In [ ]:
bh_result = None
if REAL_OHLCV_LOADED:
    bh_result = run_portfolio_backtest(
        model=BuyAndHoldBaseline(),
        features_by_symbol=features_by_sym,
        labels_by_symbol=labels_by_sym,
        prices_by_symbol=prices_by_sym,
        train_window=504,
        test_window=63,
        step=63,
        label_horizon=HORIZON,
        embargo=3,
    )
    print(f'BuyAndHold OOS Sharpe: {bh_result.oos_metrics["sharpe"]:+.3f}')


In [ ]:
if arima_result is not None and bh_result is not None:
    dm = regime_dm_test(
        arima_result.oos_forecast_errors,
        bh_result.oos_forecast_errors,
        era_labels,
        alternative='less',  # H1: ARIMA errors smaller than BH errors
    )
    print('Per-regime DM test (ARIMA vs BuyAndHold):')
    for regime, res in dm.items():
        if res is None:
            print(f'  {regime:<12} (insufficient data or degenerate)')
        else:
            print(f'  {regime:<12} stat={res.statistic:+.3f}  p={res.p_value:.4f}  n={res.n_obs}')


In [ ]:
if arima_result is not None and bh_result is not None:
    # Reuse the gate report API with ARIMA in the GBM slot to show the shape
    # of the output. Semantically the gate isn't designed for ARIMA-vs-BH;
    # this only confirms the function's output structure.
    report = phase4a_gate_report(arima_result, bh_result, era_labels)
    print(f'gate_passed:     {report["gate_passed"]}')
    print(f'pass_count:      {report["pass_count"]} of {len(report["regimes_required"])}')
    print(f'regimes_required {report["regimes_required"]}')
    print()
    print('Per-regime breakdown:')
    for regime, row in report['per_regime'].items():
        dm_str = 'n/a' if row['dm_p_value'] is None else f'{row["dm_p_value"]:.4f}'
        print(f'  {regime:<12} A.Sharpe={row["gbm_sharpe"]:+.3f}  '
              f'B.Sharpe={row["arima_sharpe"]:+.3f}  '
              f'beats={row["gbm_beats_arima"]!s:<5}  '
              f'dm_p={dm_str:<8}  n_bars={row["n_bars"]}')


---
## 9 — Cross-axis demo — era × volatility

Era and volatility regimes are orthogonal — a single date can be both `covid`
and `high_vol`, or `rate_cycle` and `low_vol`. For Milestone 1 we keep the axes
*separate*; combining them into composite labels (e.g., `covid|high_vol`) belongs
to Milestone 2/3 where regime-aware *features* enter the model.


In [ ]:
if arima_result is not None and REAL_VIX_LOADED:
    oos_idx = arima_result.oos_returns.index
    vix_on_oos = real_vix.reindex(oos_idx).ffill()  # forward-fill weekend gaps
    vol_axis = tag_regimes(oos_idx, VIXThresholdDetector(vix_on_oos))
    era_axis = tag_regimes(oos_idx, DateRangeDetector())

    cross = pd.crosstab(era_axis, vol_axis).rename_axis(index='era', columns='vol')
    print('Cross-tabulation of (era, vol) regime labels on the OOS index:')
    display(cross)

    print('\nPer-vol-regime Sharpe (ARIMA OOS):')
    for regime, m in compute_regime_metrics(arima_result.oos_returns, vol_axis).items():
        print(f'  {regime:<10} Sharpe={m["sharpe"]:+.3f}  n_bars={(vol_axis == regime).sum()}')
elif arima_result is not None:
    print('Real VIX not loaded - cross-axis demo skipped.')


---
## 10 — Mini-Milestone-6 preview — quick GBM real-data slice

A *very* truncated GBM run on the same demo panel, just to demonstrate that
`phase4a_gate_report` semantically does what the PRD specifies when the inputs
are GBM and ARIMA on real data. **This is not the Phase 4A gate evaluation** —
that requires the full universe, more folds, and the full feature set from
Milestones 2–3. Here we use the existing 17-feature Phase 2.5 set.

Expect this cell to run for ~1–3 minutes. Skip if iterating on §1–§9.


In [ ]:
RUN_GBM_PREVIEW = True   # set False to skip the slower GBM run

gbm_result = None
if RUN_GBM_PREVIEW and REAL_OHLCV_LOADED:
    from quant.features.engineering import build_features
    from quant.models.gbm import GBMModel

    # build_features is a bulk API: takes symbols + prices_by_symbol,
    # returns {symbol: feature_df}. FRED ASOF join happens in one DuckDB query.
    try:
        feat_by_sym = build_features(
            symbols=list(prices_by_sym.keys()),
            prices_by_symbol=prices_by_sym,
            sentiment_df=None,
        )
    except Exception as exc:
        print(f'build_features failed ({type(exc).__name__}): {exc}')
        feat_by_sym = {}

    gbm_features_by_sym, gbm_labels_by_sym, gbm_prices_by_sym = {}, {}, {}
    for s, X in feat_by_sym.items():
        # build_features returns UTC tz-aware indexes (FRED ASOF join injects
        # UTC); our prices_by_sym is tz-naive. Strip tz so indexes intersect.
        if getattr(X.index, 'tz', None) is not None:
            X = X.copy()
            X.index = X.index.tz_localize(None)
        X_clean = X.dropna()
        fwd = (prices_by_sym[s]['close'].shift(-HORIZON) / prices_by_sym[s]['close'] - 1.0).dropna()
        common = X_clean.index.intersection(fwd.index)
        if len(common) < 600:
            continue
        gbm_features_by_sym[s] = X_clean.loc[common]
        gbm_labels_by_sym[s] = fwd.loc[common]
        gbm_prices_by_sym[s] = prices_by_sym[s].loc[common]
    print(f'Built GBM inputs for {len(gbm_features_by_sym)} symbols')

    if gbm_features_by_sym:
        # Trim n_iter to keep this preview fast - full evaluation uses n_iter=50.
        gbm_result = run_portfolio_backtest(
            model=GBMModel(n_iter=10, n_splits=3, random_state=11),
            features_by_symbol=gbm_features_by_sym,
            labels_by_symbol=gbm_labels_by_sym,
            prices_by_symbol=gbm_prices_by_sym,
            train_window=504,
            test_window=63,
            step=63,
            label_horizon=HORIZON,
            embargo=3,
        )
        print(f'GBM OOS Sharpe (preview, n_iter=10): {gbm_result.oos_metrics["sharpe"]:+.3f}')


In [ ]:
if gbm_result is not None and arima_result is not None:
    # Re-tag the GBM OOS index - may differ from ARIMA after the 252-bar
    # feature warmup drops the first ~year of each symbol's history.
    gbm_era = tag_regimes(gbm_result.oos_returns.index, DateRangeDetector())
    arima_era = tag_regimes(arima_result.oos_returns.index, DateRangeDetector())

    print('Per-regime Sharpe - GBM vs ARIMA on the same demo slice:')
    gbm_per = compute_regime_metrics(gbm_result.oos_returns, gbm_era)
    arima_per = compute_regime_metrics(arima_result.oos_returns, arima_era)
    all_regimes = sorted(set(gbm_per) | set(arima_per))
    print(f'  {"regime":<12} {"GBM":>8} {"ARIMA":>8} {"delta":>8}')
    for r in all_regimes:
        g = gbm_per.get(r, {}).get('sharpe', float('nan'))
        a = arima_per.get(r, {}).get('sharpe', float('nan'))
        delta = g - a
        print(f'  {r:<12} {g:>+8.3f} {a:>+8.3f} {delta:>+8.3f}')

    # Gate report needs matching indices for the DM test. Align first.
    common_idx = gbm_result.oos_returns.index.intersection(arima_result.oos_returns.index)
    if len(common_idx) > 0:
        gbm_aligned = BacktestResult(
            oos_metrics=gbm_result.oos_metrics,
            is_metrics=gbm_result.is_metrics,
            equity_curve=gbm_result.equity_curve,
            trade_log=gbm_result.trade_log,
            oos_returns=gbm_result.oos_returns.loc[common_idx],
            oos_forecast_errors=gbm_result.oos_forecast_errors.loc[common_idx],
        )
        arima_aligned = BacktestResult(
            oos_metrics=arima_result.oos_metrics,
            is_metrics=arima_result.is_metrics,
            equity_curve=arima_result.equity_curve,
            trade_log=arima_result.trade_log,
            oos_returns=arima_result.oos_returns.loc[common_idx],
            oos_forecast_errors=arima_result.oos_forecast_errors.loc[common_idx],
        )
        common_era = tag_regimes(common_idx, DateRangeDetector())
        gate = phase4a_gate_report(gbm_aligned, arima_aligned, common_era)
        print('\nPreview gate verdict (NOT THE MILESTONE-6 GATE):')
        print(f'  gate_passed: {gate["gate_passed"]}  pass_count: {gate["pass_count"]}')
        print(f'  dm p-values: {gate["dm_p_values"]}')


---
## 11 — Harness self-tests visualized

These mirror assertions already in the test suite, but visible to the eye so a
reader of the notebook can confirm the harness is not silently broken:

- **Random model** → \|Sharpe\| < ~1.5 in every regime (no systematic edge).
- **Perfect-foresight model** → Sharpe ≫ 0 in every regime (real edge transmits).

If either fails, the harness is wrong, not the test.


In [ ]:
# Reuse the stub models from the test suite verbatim - keeps notebook and
# test semantics identical.
class RandomModel:
    def __init__(self, seed=42):
        self._rng = np.random.default_rng(seed)
    def fit(self, X, y): return self
    def predict(self, X): return self._rng.standard_normal(len(X))

class PerfectForesightModel:
    def fit(self, X, y): return self
    def predict(self, X): return X[:, -1]


In [ ]:
# Build a small synthetic panel for self-tests (no lake dependency).
N_SELF = 600
self_dates = pd.bdate_range('2018-01-02', periods=N_SELF)
self_syms = ['SYNTH1', 'SYNTH2']

def _make_synth_prices(seed):
    r = np.random.default_rng(seed)
    close = 100.0 * np.exp(np.cumsum(r.normal(0.0, 0.01, N_SELF)))
    open_ = close * (1 + r.uniform(-0.002, 0.002, N_SELF))
    return pd.DataFrame({
        'open': open_, 'high': np.maximum(close, open_) * 1.005,
        'low':  np.minimum(close, open_) * 0.995, 'close': close,
        'volume': r.integers(500_000, 2_000_000, N_SELF).astype(float),
    }, index=self_dates)

self_prices = {s: _make_synth_prices(seed=i) for i, s in enumerate(self_syms)}
self_labels = {s: (p['close'].shift(-1) / p['close'] - 1.0) for s, p in self_prices.items()}
self_features = {
    s: pd.DataFrame(np.random.default_rng(i).standard_normal((N_SELF, 5)), index=self_dates)
    for i, s in enumerate(self_syms)
}

def synth_regime_split(idx):
    midpoint = idx[len(idx) // 2]
    return pd.Series(np.where(idx < midpoint, 'first_half', 'second_half'), index=idx)


In [ ]:
# Random model - must show ~0 Sharpe per regime.
rand_result = run_portfolio_backtest(
    model=RandomModel(seed=7),
    features_by_symbol=self_features,
    labels_by_symbol=self_labels,
    prices_by_symbol=self_prices,
    train_window=200, test_window=50, step=50, label_horizon=1, embargo=3,
)
rand_labels = synth_regime_split(rand_result.oos_returns.index)
rand_per = compute_regime_metrics(rand_result.oos_returns, rand_labels)
print('Random model - per-regime Sharpe:')
for r, m in rand_per.items():
    print(f'  {r:<14} Sharpe={m["sharpe"]:+.3f}')
    assert abs(m['sharpe']) < 1.5, f'Random model regime {r} Sharpe too large'
print('\n✓ Random model passes the per-regime self-test.')


In [ ]:
# Perfect-foresight model - must show high Sharpe per regime.
pf_features = {}
for s, base in self_features.items():
    augmented = base.copy()
    p = self_prices[s]
    traded_ret = (p['open'].shift(-2) / p['open'].shift(-1) - 1.0).fillna(0.0)
    augmented['pf_signal'] = traded_ret.values
    pf_features[s] = augmented

pf_result = run_portfolio_backtest(
    model=PerfectForesightModel(),
    features_by_symbol=pf_features,
    labels_by_symbol=self_labels,
    prices_by_symbol=self_prices,
    train_window=200, test_window=50, step=50, label_horizon=1, embargo=3,
    commission_per_share=0.0, slippage_bps=0.0,
)
pf_labels = synth_regime_split(pf_result.oos_returns.index)
pf_per = compute_regime_metrics(pf_result.oos_returns, pf_labels)
print('Perfect-foresight - per-regime Sharpe:')
for r, m in pf_per.items():
    print(f'  {r:<14} Sharpe={m["sharpe"]:+.3f}')
    assert m['sharpe'] > 0.5, f'Perfect-foresight regime {r} Sharpe too small'
print('\n✓ Perfect-foresight model passes the per-regime self-test.')


---
## 12 — What's next (Milestones 2–6)

Milestone 1 ships the *evaluation substrate*. Subsequent milestones write to it:

| # | Milestone | How it uses Milestone 1 |
|---|---|---|
| 2 | Label-scheme ablation matrix | Each label scheme produces a `BacktestResult`; `phase4a_gate_report` ranks them per-regime |
| 3 | Cross-sectional + regime-aware features + per-feature ablation | Each feature toggle produces a result; per-regime Sharpe attributes the lift to a regime |
| 4 | Feature catalog (YAML) | Catalog entries store per-regime metrics emitted by `compute_regime_metrics` |
| 5 | FRED ASOF-join leakage investigation | Regime-conditional Sharpe per FRED feature exposes look-ahead artifacts |
| 6 | Phase 4A exit-gate report | The actual gate run on the full Dow 30 + ETF panel; uses this exact `phase4a_gate_report` API |

**Important:** the gate verdict in §10 above is a *preview only* — it uses a
truncated GBM (n_iter=10), a 5-symbol panel, and the unchanged Phase 2.5 feature
set. The real Milestone 6 evaluation runs after Milestones 2–3 have actually
redesigned the labels and features, on the full universe, with the full
hyperparameter search. Treat the preview output as plumbing confirmation, not
as evidence about edge.
